In [ ]:
import gpytoolbox as gpy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.collections as mc
import matplotlib.patches as mp
import trimesh

import dvx_ext

# 3D Voxelization Functions Overview

## Helper Functions

In [ ]:
mesh = trimesh.load_mesh("hand.obj")
V, F = np.array(mesh.vertices, dtype=np.float32), np.array(mesh.faces, dtype=np.int32)

# Global parameters
GRID_SIZE = (4, 4, 4)
FILTER_RADIUS = (2 / GRID_SIZE[0]) / 2
NUM_SAMPLES_PER_VOXEL = 64
NUM_SAMPLES_PER_SIMPLEX = 64
FD_EPSILON = 1e-4

print(f"Mesh info: {len(V)} vertices, {len(F)} faces")
print(f"Grid size: {GRID_SIZE}")
print(f"Filter radius: {FILTER_RADIUS}")

In [ ]:
def compute_jacobian_column_via_backward(primal_func, backward_func, v, f, dtype, vertex_idx=0, 
                                          primal_kwargs={}, backward_kwargs={}):
    """
    Compute the Jacobian column for a single vertex by calling backward once per output.
    
    For each output element, we set d_L to a one-hot vector and call backward.
    This gives us d_output[i,j,k] / d_vertex[vertex_idx, :] for all outputs.
    """
    # Compute primal
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)
    primal_func(v, f, occupancy, **primal_kwargs)
    
    num_outputs = np.prod(GRID_SIZE)
    grad_wrt_vertex = np.zeros((*GRID_SIZE, 3), dtype=dtype)
    
    for out_idx in range(num_outputs):
        # Create one-hot d_L
        d_L = np.zeros(GRID_SIZE, dtype=dtype)
        d_L.flat[out_idx] = 1.0
        
        # Call backward
        d_vertices = np.zeros_like(v, dtype=dtype)
        backward_func(v, f, occupancy, d_vertices, d_L, **backward_kwargs)
        
        # Extract gradient for the specified vertex and store at the output location
        out_coords = np.unravel_index(out_idx, GRID_SIZE)
        grad_wrt_vertex[out_coords] = d_vertices[vertex_idx, :]
    
    return grad_wrt_vertex


def compute_jacobian_column_via_forward(primal_func, forward_func, v, f, dtype, vertex_idx=0,
                                         primal_kwargs={}, forward_kwargs={}):
    """
    Compute the Jacobian column for a single vertex by calling forward AD once per coordinate.
    
    For each coordinate (x, y, z), we set d_vertices to a one-hot vector and call forward.
    This gives us d_output / d_vertex[vertex_idx, coord] for all outputs.
    """
    # Compute primal
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)
    primal_func(v, f, occupancy, **primal_kwargs)
    
    grad_wrt_vertex = np.zeros((*GRID_SIZE, 3), dtype=dtype)
    
    for coord in range(3):
        # Create one-hot d_vertices
        d_vertices = np.zeros_like(v, dtype=dtype)
        d_vertices[vertex_idx, coord] = 1.0
        
        # Call forward AD
        d_occupancy = np.zeros(GRID_SIZE, dtype=dtype)
        forward_func(v, f, occupancy, d_vertices, d_occupancy, **forward_kwargs)
        
        grad_wrt_vertex[..., coord] = d_occupancy
    
    return grad_wrt_vertex


def compute_jacobian_column_via_fd(primal_func, v, f, dtype, vertex_idx=0, eps=1e-4, primal_kwargs={}):
    """
    Compute the Jacobian column for a single vertex using finite differences.
    
    For each coordinate (x, y, z) of the vertex, perturb and measure change in all outputs.
    """
    grad_wrt_vertex = np.zeros((*GRID_SIZE, 3), dtype=dtype)
    
    for coord in range(3):
        v_plus = v.copy()
        v_plus[vertex_idx, coord] += eps
        occupancy_plus = np.zeros(GRID_SIZE, dtype=dtype)
        primal_func(v_plus, f, occupancy_plus, **primal_kwargs)
        
        v_minus = v.copy()
        v_minus[vertex_idx, coord] -= eps
        occupancy_minus = np.zeros(GRID_SIZE, dtype=dtype)
        primal_func(v_minus, f, occupancy_minus, **primal_kwargs)
        
        grad_wrt_vertex[..., coord] = (occupancy_plus - occupancy_minus) / (2 * eps)
    
    return grad_wrt_vertex

In [ ]:
# Compare Jacobian columns for first vertex
v = V.astype(np.float64)
f = F.astype(np.uint32)
mc_forward_kwargs = {'num_samples_per_simplex': NUM_SAMPLES_PER_SIMPLEX, 'filter_radius': FILTER_RADIUS}
mc_primal_kwargs = {'num_samples_per_voxel': NUM_SAMPLES_PER_VOXEL, 'filter_radius': FILTER_RADIUS}
cf_forward_kwargs = {}
cf_primal_kwargs = {}

# Using cf method (no extra kwargs)
jac_forward = compute_jacobian_column_via_forward(
    dvx_ext.voxelize_cf_f64, dvx_ext.voxelize_forward_cf_f64,
    v, f, np.float64, vertex_idx=0, primal_kwargs=cf_primal_kwargs, forward_kwargs=cf_forward_kwargs
)

jac_backward = compute_jacobian_column_via_backward(
    dvx_ext.voxelize_cf_f64, dvx_ext.voxelize_backward_cf_f64,
    v, f, np.float64, vertex_idx=0, primal_kwargs=cf_primal_kwargs, backward_kwargs=cf_forward_kwargs
)

jac_fd = compute_jacobian_column_via_fd(
    dvx_ext.voxelize_cf_f64, v, f, np.float64, vertex_idx=0, eps=FD_EPSILON, primal_kwargs=cf_primal_kwargs
)

# print("Jacobian column:")
# print(f"  Forward:  {jac_forward}")
# print(f"  Backward: {jac_backward}")
# print(f"  FD:       {jac_fd}")

print("\n--- Forward AD vs FD ---")
print(f"  Max abs diff: {np.abs(jac_forward - jac_fd).max():.6e}")
print(f"  Relative error: {np.linalg.norm(jac_forward - jac_fd) / (np.linalg.norm(jac_fd) + 1e-10):.6e}")

print("\n--- Backward AD vs FD ---")
print(f"  Max abs diff: {np.abs(jac_backward - jac_fd).max():.6e}")
print(f"  Relative error: {np.linalg.norm(jac_backward - jac_fd) / (np.linalg.norm(jac_fd) + 1e-10):.6e}")

print("\n--- Forward AD vs Backward AD ---")
print(f"  Max abs diff: {np.abs(jac_forward - jac_backward).max():.6e}")
print(f"  Relative error: {np.linalg.norm(jac_forward - jac_backward) / (np.linalg.norm(jac_forward) + 1e-10):.6e}")

# Check where backward differs most from FD
diff = np.abs(jac_backward - jac_fd)
max_idx = np.unravel_index(np.argmax(diff), diff.shape)
print(f"\nMax backward-FD difference at index {max_idx}:")
print(f"  Forward:  {jac_forward[max_idx]:.6e}")
print(f"  Backward: {jac_backward[max_idx]:.6e}")
print(f"  FD:       {jac_fd[max_idx]:.6e}")